Moving models and MongoDB data to PostgreSQL database

In [5]:
from pymongo import MongoClient

from dotenv import load_dotenv
import os

import psycopg

In [6]:
load_dotenv('../src/credentials/.env')
mongo_url = os.getenv("MONGO_CONNECTION_STRING")
postgres_url = os.getenv("DATABASE_URL")

In [7]:
# INSERT queries
def get_connection():
    return psycopg.connect(postgres_url)

def add_model_version(version_name, model_path, notes=None):
    query = """
        INSERT INTO models (version_name, model_path, notes)
        VALUES (%s, %s, %s)
        RETURNING id;
    """

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (version_name, model_path, notes))
            model_id = cur.fetchone()[0]
            return model_id
        
def add_prediction(true_label, predicted_label, image_b64, was_correct, confidence, version_name):
    query = """
        INSERT INTO predictions (
            true_label, 
            predicted_label, 
            image_b64, 
            was_correct, 
            confidence, 
            model_id
        )
        VALUES (
            %s, %s, %s, %s, %s, 
            (SELECT id FROM models WHERE version_name = %s)
        )
        RETURNING id;
    """

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (true_label, predicted_label, image_b64, was_correct, confidence, version_name))
            prediction_id = cur.fetchone()[0]
            return prediction_id

In [8]:
# INSERT models
add_model_version('char_cnn', 'models/char_cnn.keras', 'trained with initial dataset')
add_model_version('char_cnn_feedback', 'models/char_cnn_feedback.keras', 'trained with initial dataset plus 15x62 user-generated samples')

2

In [14]:
client = MongoClient(mongo_url)
db = client["classifier"]
collection = db["feedback"]

In [17]:
# INSERT predictions

# !!IMPORTANT!!:
# I originally didn't store confidence value so this entry will be NULL for these 930 samples
data = collection.find({})
for record in data:
    add_prediction(
        true_label = record["true_label"],
        predicted_label = record["predicted_label"],
        image_b64 = record["image_b64"],
        was_correct = record["was_correct"],
        confidence = None,
        version_name = "char_cnn"
    )